In [1]:
import os

In [2]:
%pwd

'c:\\Users\\MANJU\\OneDrive\\Desktop\\MajorProjectResearchPaper\\EndToEndMajorProject\\research'

In [3]:
os.chdir("C:/Users/MANJU/OneDrive/Desktop/MajorProjectResearchPaper/EndToEndMajorProject")

In [4]:
%pwd

'C:\\Users\\MANJU\\OneDrive\\Desktop\\MajorProjectResearchPaper\\EndToEndMajorProject'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    base_data_path: Path
    transformed_data_path: Path

In [6]:
from src.cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath= CONFIG_FILE_PATH,
        params_filepath= PARAMS_FILE_PATH):

        self.config= read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            base_data_path=Path(config.base_data_path),
            transformed_data_path=Path(config.transformed_data_path)
        )

        return data_transformation_config

In [8]:
import os
import shutil
import pandas as pd
from pathlib import Path

In [9]:
class DataTransformation:
    def __init__(self, config):
        self.config = config

    def organize_dataset(self, csv_file, image_folder):
        df = pd.read_csv(csv_file)

        for _, row in df.iterrows():
            img_id = row['id_code']
            label = str(row['diagnosis'])

            possible_paths = [
                os.path.join(image_folder, img_id + ".png"),
                os.path.join(image_folder, img_id + ".jpg")
            ]

            src = None
            for path in possible_paths:
                if os.path.exists(path):
                    src = path
                    break

            if src is None:
                print(f"❌ Not found: {img_id}")
                continue

            dst_folder = os.path.join(self.config.transformed_data_path, label)
            os.makedirs(dst_folder, exist_ok=True)

            shutil.copy(src, os.path.join(dst_folder, os.path.basename(src)))

    def run(self):
        base_path = self.config.base_data_path

        self.organize_dataset(
            os.path.join(base_path, "train_1.csv"),
            os.path.join(base_path, "train_images", "train_images")
        )

        self.organize_dataset(
            os.path.join(base_path, "test.csv"),
            os.path.join(base_path, "test_images", "test_images")
        )

        self.organize_dataset(
            os.path.join(base_path, "valid.csv"),
            os.path.join(base_path, "val_images", "val_images")
        )

        print("✅ Data Transformation Completed")

In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(data_transformation_config)
    data_transformation.run()
except Exception as e:
    raise e

[2026-04-02 00:25:21,397: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-02 00:25:21,409: INFO: common: yaml file: params.yaml loaded successfully]


[2026-04-02 00:25:21,420: INFO: common: created directory at:artifacts]
[2026-04-02 00:25:21,429: INFO: common: created directory at:artifacts/data_transformation]
✅ Data Transformation Completed
